# 🛡️ Veritas AI: 3-Class EfficientNetB0 Image Authenticity Classifier Training

This notebook trains a production-grade 3-class deepfake authenticity model (`ai_generated`, `ai_modified`, `real`) on Google Colab with **GPU hardware acceleration**, **high-speed local disk caching**, and **compiled `@tf.function` GPU graph steps**.

### **Required Google Drive Files**
Ensure your Google Drive has the folder `/content/drive/MyDrive/deepfake_project/datasets/` containing:
- `processed_3class.zip`
- `image_detection.zip`
- `genimage_extracted.zip`
- `manifest_combined.csv` (or `manifest_combined.csv.zip`)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# Paths configuration
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/deepfake_project/datasets')
OUTPUT_DIR = Path('/content/drive/MyDrive/deepfake_project/outputs')
BASE_DATA_DIR = Path('/content/datasets')  # High-speed local Colab disk

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"[*] Google Drive Data Dir : {DRIVE_DATA_DIR}")
print(f"[*] Colab Local Disk Root : {BASE_DATA_DIR}")
print(f"[*] Persistent Output Dir : {OUTPUT_DIR}")

## 2. Extract Datasets to Dedicated Local Subfolders
Safely extracts each zip directly into its target subfolder (`processed_3class/`, `image_detection/`, `genimage_extracted/`), handles `manifest_combined.csv` / `manifest_combined.csv.zip`, and validates file counts.

In [ ]:
import time
import zipfile
import os
import shutil
from pathlib import Path

unzip_start_time = time.time()
print("=" * 80)
print("[*] EXTRACTING DATASETS TO TARGET SUBFOLDERS (/content/datasets/<subfolder>/)")
print("=" * 80)

dataset_configs = {
    "processed_3class": {
        "zip_name": "processed_3class.zip",
        "target_dir": BASE_DATA_DIR / "processed_3class",
        "expected_files": 45001
    },
    "image_detection": {
        "zip_name": "image_detection.zip",
        "target_dir": BASE_DATA_DIR / "image_detection",
        "expected_files": 15000
    },
    "genimage_extracted": {
        "zip_name": "genimage_extracted.zip",
        "target_dir": BASE_DATA_DIR / "genimage_extracted",
        "expected_files": 11251
    }
}

# 1. Extract each zip specifically into its dedicated subfolder
for key, meta in dataset_configs.items():
    zip_name = meta["zip_name"]
    zip_path = DRIVE_DATA_DIR / zip_name
    target_dir = meta["target_dir"]
    
    target_dir.mkdir(parents=True, exist_ok=True)
    
    if zip_path.exists():
        print(f"[*] Extracting {zip_name} directly into {target_dir}...")
        t0 = time.time()
        !unzip -q -o "{zip_path}" -d "{target_dir}"
        elapsed = time.time() - t0
        print(f"    -> Extracted in {elapsed:.1f}s")
        
        nested_sub = target_dir / key
        if nested_sub.exists() and nested_sub.is_dir():
            print(f"    [!] Detected nested wrapping folder {nested_sub}, flattening to {target_dir}...")
            for item in os.listdir(nested_sub):
                src_item = nested_sub / item
                dst_item = target_dir / item
                if not dst_item.exists():
                    shutil.move(str(src_item), str(target_dir))
            try:
                nested_sub.rmdir()
            except Exception:
                pass
    elif any(target_dir.iterdir()):
        print(f"[*] Found existing files in target directory: {target_dir}")
    else:
        drive_folder = DRIVE_DATA_DIR / key
        if drive_folder.exists():
            print(f"[*] Zip not found. Copying uncompressed folder from Drive {drive_folder} to {target_dir}...")
            !cp -r "{drive_folder}/." "{target_dir}/"
        else:
            print(f"[!] Warning: Neither {zip_path} nor {drive_folder} was found on Google Drive!")

# 2. Extract / Copy manifest_combined.csv
print("\n--- Manifest Setup ---")
drive_manifest_csv = DRIVE_DATA_DIR / "manifest_combined.csv"
drive_manifest_zip = DRIVE_DATA_DIR / "manifest_combined.csv.zip"
local_manifest_csv = BASE_DATA_DIR / "manifest_combined.csv"

if drive_manifest_zip.exists():
    print(f"[*] Found zipped manifest: {drive_manifest_zip}")
    !unzip -q -o "{drive_manifest_zip}" -d "{BASE_DATA_DIR}"
    if not local_manifest_csv.exists():
        found = list(BASE_DATA_DIR.glob("*manifest_combined*.csv"))
        if found:
            shutil.move(str(found[0]), str(local_manifest_csv))
    if local_manifest_csv.exists():
        print(f"    -> Successfully extracted to {local_manifest_csv}")
elif drive_manifest_csv.exists():
    print(f"[*] Copying plain manifest from Drive: {drive_manifest_csv}")
    shutil.copy2(drive_manifest_csv, local_manifest_csv)
    print(f"    -> Successfully copied to {local_manifest_csv}")
else:
    alt_manifests = list(DRIVE_DATA_DIR.glob("*manifest*.zip")) + list(DRIVE_DATA_DIR.glob("*manifest*.csv"))
    if alt_manifests:
        chosen = alt_manifests[0]
        if chosen.suffix == ".zip":
            !unzip -q -o "{chosen}" -d "{BASE_DATA_DIR}"
        else:
            shutil.copy2(chosen, local_manifest_csv)
        print(f"[*] Used fallback manifest {chosen} -> {local_manifest_csv}")
    else:
        print(f"[!] ERROR: manifest_combined.csv (or .csv.zip) not found on Google Drive ({DRIVE_DATA_DIR})!")

# 3. Verify extracted file counts
print("\n--- Verification of Extracted Dataset Files ---")
for key, meta in dataset_configs.items():
    target_folder = meta["target_dir"]
    expected = meta["expected_files"]
    if target_folder.exists():
        actual_count = sum(len(files) for _, _, files in os.walk(target_folder))
        status = "✅ MATCH" if actual_count >= expected else "⚠️ MISMATCH / PARTIAL"
        print(f"  * {key:<22}: {actual_count:,} files (Expected: {expected:,}) -> {status}")
    else:
        print(f"  * {key:<22}: [MISSING DIRECTORY]")

if local_manifest_csv.exists():
    with open(local_manifest_csv, "r", encoding="utf-8") as f:
        line_count = sum(1 for _ in f) - 1
    print(f"\n[*] Manifest verified: {local_manifest_csv} ({line_count:,} sample rows, {local_manifest_csv.stat().st_size / (1024*1024):.2f} MB)")

total_unzip_time = time.time() - unzip_start_time
print(f"\n[*] Total Dataset Setup Time on Local Disk: {total_unzip_time:.1f}s ({total_unzip_time/60:.2f} mins)")
print("=" * 80)

## 3. Install Required Dependencies

In [ ]:
!pip install -q opencv-python-headless mediapipe scikit-learn matplotlib seaborn pandas numpy

## 4. Hardware Verification & GPU Device Confirmation

In [ ]:
import tensorflow as tf

print(f"[*] TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"[*] Detected GPUs: {gpus}")

if not gpus:
    raise SystemError(
        "⚠️ WARNING: No GPU detected! Please change Colab Runtime to GPU "
        "(Menu -> Runtime -> Change runtime type -> Hardware accelerator: T4 GPU) before training."
    )
else:
    print(f"✅ GPU Hardware Active: {gpus[0].name}")
    !nvidia-smi

## 5. Model Building with Explicit GPU Device Context & Device Placement Audit
Guarantees that model weights and variables are explicitly created on `/GPU:0` and compiled with `@tf.function`.

In [ ]:
import time
import random
import json
import csv
import threading
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

CLASS_TO_IDX = {"ai_generated": 0, "ai_modified": 1, "real": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_WEIGHTS_LIST = [1.45, 1.52, 1.00]
CLASS_WEIGHTS_T = tf.constant(CLASS_WEIGHTS_LIST, dtype=tf.float32)
CLASS_WEIGHTS = {0: 1.45, 1: 1.52, 2: 1.00}

def get_var_device(var):
    """Helper to read device location across TF 2.x and Keras 3."""
    if hasattr(var, "device"): return var.device
    if hasattr(var, "value") and hasattr(var.value, "device"): return var.value.device
    if hasattr(var, "handle") and hasattr(var.handle, "device"): return var.handle.device
    return "unknown"

# Thread-local OpenCV Cascade for thread-safe parallel tf.data loading
_thread_local = threading.local()
def get_face_cascade():
    if not hasattr(_thread_local, "cascade"):
        _thread_local.cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    return _thread_local.cascade

def resolve_image_path(base_dir: Path, rel_path: str) -> Path:
    p = base_dir / rel_path
    if p.exists(): return p
    if "image_detection" in rel_path:
        p2 = base_dir.parent / "dataset" / rel_path
        if p2.exists(): return p2
    return p

def load_and_preprocess_image(
    base_data_dir: Path,
    image_rel_path: str,
    source_dataset: str,
    is_training: bool = False,
    target_size: tuple = (224, 224)
) -> np.ndarray:
    abs_path = resolve_image_path(base_data_dir, image_rel_path)
    if not abs_path.exists():
        return np.zeros((target_size[0], target_size[1], 3), dtype=np.float32)

    img_bgr = cv2.imread(str(abs_path))
    if img_bgr is None:
        return np.zeros((target_size[0], target_size[1], 3), dtype=np.float32)

    h, w, _ = img_bgr.shape

    # Branch 1: FaceForensics face detection with downsampled thumbnail acceleration
    if "faceforensics" in source_dataset.lower():
        face_cascade = get_face_cascade()
        scale = 320.0 / max(h, w)
        if scale < 1.0:
            thumb_w = int(w * scale)
            thumb_h = int(h * scale)
            thumb_bgr = cv2.resize(img_bgr, (thumb_w, thumb_h), interpolation=cv2.INTER_NEAREST)
            gray = cv2.cvtColor(thumb_bgr, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(20, 20))
            inv_scale = 1.0 / scale
            scaled_faces = [[int(x * inv_scale), int(y * inv_scale), int(fw * inv_scale), int(fh * inv_scale)] for x, y, fw, fh in faces]
        else:
            gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
            scaled_faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(30, 30))

        if len(scaled_faces) > 0:
            fx, fy, fw, fh = max(scaled_faces, key=lambda b: b[2] * b[3])
            margin_x = int(0.20 * fw)
            margin_y = int(0.20 * fh)
            x1 = max(0, fx - margin_x)
            y1 = max(0, fy - margin_y)
            x2 = min(w, fx + fw + margin_x)
            y2 = min(h, fy + fh + margin_y)
            cropped = img_bgr[y1:y2, x1:x2]
        else:
            min_dim = min(h, w)
            sy, sx = (h - min_dim) // 2, (w - min_dim) // 2
            cropped = img_bgr[sy:sy + min_dim, sx:sx + min_dim]
        resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)
    else:
        resized = cv2.resize(img_bgr, target_size, interpolation=cv2.INTER_LINEAR)

    if is_training:
        if random.random() > 0.5:
            resized = cv2.flip(resized, 1)
        angle = random.uniform(-10, 10)
        M = cv2.getRotationMatrix2D((target_size[0] / 2, target_size[1] / 2), angle, 1.0)
        resized = cv2.warpAffine(resized, M, target_size, borderMode=cv2.BORDER_REFLECT)
        quality = random.randint(70, 95)
        _, enc = cv2.imencode('.jpg', resized, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        resized = cv2.imdecode(enc, 1)
        alpha, beta = random.uniform(0.9, 1.1), random.uniform(-15, 15)
        resized = cv2.convertScaleAbs(resized, alpha=alpha, beta=beta)

    img_rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    return img_rgb.astype(np.float32)

def create_tf_data_pipeline(df: pd.DataFrame, base_data_dir: Path, batch_size: int = 64, is_training: bool = False) -> tf.data.Dataset:
    image_paths = df["image_path"].values.astype(str)
    source_datasets = df["source_dataset"].values.astype(str)
    labels = df["class"].map(CLASS_TO_IDX).values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, source_datasets, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=min(len(df), 4096), reshuffle_each_iteration=True)

    def _py_loader(p_bytes, s_bytes, is_train_bool):
        p_str = p_bytes.numpy().decode("utf-8")
        s_str = s_bytes.numpy().decode("utf-8")
        return load_and_preprocess_image(base_data_dir, p_str, s_str, is_training=bool(is_train_bool))

    def _map_fn(p, s, l):
        img = tf.py_function(func=_py_loader, inp=[p, s, is_training], Tout=tf.float32)
        img.set_shape([224, 224, 3])
        l.set_shape([])
        return img, l

    ds = ds.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

def build_model():
    with tf.device('/GPU:0'):
        base_model = tf.keras.applications.EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
        x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(base_model.output)
        x = tf.keras.layers.Dense(256, activation="relu", name="dense_256")(x)
        x = tf.keras.layers.Dropout(0.3, name="dropout_0.3")(x)
        outputs = tf.keras.layers.Dense(3, activation="softmax", name="predictions")(x)
        model = tf.keras.Model(inputs=base_model.input, outputs=outputs, name="EfficientNetB0_Authenticity")
    return model, base_model

# Create and audit initial model placement
model, base_model = build_model()
var0 = model.trainable_variables[0]
var0_dev = get_var_device(var0)

print("=" * 80)
print(f"[*] MODEL INITIALIZATION & DEVICE PLACEMENT AUDIT:")
print(f"    - Variable 0 Name   : {var0.name}")
print(f"    - Variable 0 Device : {var0_dev}")
print(f"    - Total Variables   : {len(model.trainable_variables)}")
print("=" * 80)

if "GPU" not in var0_dev.upper():
    raise RuntimeError(f"[ERROR] Model weights are on {var0_dev} instead of GPU! Check Colab runtime settings.")
else:
    print("✅ SUCCESS: Model variables are verified on GPU:0!")

## 6. Sanity Check Smoke Test with Compiled `@tf.function`
Measures separate (a) Data Loading time vs (b) Compiled GPU Compute time on 600 samples.

In [ ]:
print("=" * 80)
print("[*] RUNNING BENCHMARKED SMOKE TEST (600 Samples, Compiled GPU Graph)")
print("=" * 80)

manifest_file = BASE_DATA_DIR / 'manifest_combined.csv'
full_df = pd.read_csv(manifest_file)

train_samples, val_samples, test_samples = [], [], []
for c in ("ai_generated", "ai_modified", "real"):
    train_samples.append(full_df[(full_df["split"] == "train") & (full_df["class"] == c)].sample(n=200, random_state=42))
    val_samples.append(full_df[(full_df["split"] == "validation") & (full_df["class"] == c)].sample(n=50, random_state=42))
    test_samples.append(full_df[(full_df["split"] == "test") & (full_df["class"] == c)].sample(n=50, random_state=42))

smoke_train_df = pd.concat(train_samples).sample(frac=1, random_state=42).reset_index(drop=True)
smoke_val_df = pd.concat(val_samples).sample(frac=1, random_state=42).reset_index(drop=True)
smoke_test_df = pd.concat(test_samples).sample(frac=1, random_state=42).reset_index(drop=True)

smoke_train_ds = create_tf_data_pipeline(smoke_train_df, BASE_DATA_DIR, batch_size=32, is_training=True)
smoke_val_ds = create_tf_data_pipeline(smoke_val_df, BASE_DATA_DIR, batch_size=32, is_training=False)
smoke_test_ds = create_tf_data_pipeline(smoke_test_df, BASE_DATA_DIR, batch_size=32, is_training=False)

smoke_model, smoke_base = build_model()
smoke_base.trainable = False

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
smoke_opt_p1 = tf.keras.optimizers.Adam(1e-3)
# Eagerly build optimizer slot variables before tracing
smoke_opt_p1.build(smoke_model.trainable_variables)

@tf.function
def smoke_train_step(images, labels, opt):
    with tf.GradientTape() as tape:
        preds = smoke_model(images, training=True)
        loss_raw = loss_fn(labels, preds)
        weights = tf.gather(CLASS_WEIGHTS_T, labels)
        loss = tf.reduce_mean(loss_raw * weights)
    grads = tape.gradient(loss, smoke_model.trainable_variables)
    opt.apply_gradients(zip(grads, smoke_model.trainable_variables))
    acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(preds, axis=1, output_type=tf.int32), labels), tf.float32))
    return loss, acc

# Warmup JIT trace
dummy_x = tf.zeros((32, 224, 224, 3), dtype=tf.float32)
dummy_y = tf.zeros((32,), dtype=tf.int32)
_ = smoke_train_step(dummy_x, dummy_y, smoke_opt_p1)

data_times, comp_times = [], []
ds_iter = iter(smoke_train_ds)
batch_count = int(np.ceil(len(smoke_train_df) / 32))

for b in range(batch_count):
    t_d0 = time.time()
    X_b, y_b = next(ds_iter)
    t_data = time.time() - t_d0
    data_times.append(t_data)
    
    t_c0 = time.time()
    loss, acc = smoke_train_step(X_b, y_b, smoke_opt_p1)
    t_comp = time.time() - t_c0
    comp_times.append(t_comp)
    
    print(f"  Smoke Batch [{b+1}/{batch_count}] - Data Load: {t_data*1000:5.1f}ms | GPU Compute: {t_comp*1000:5.1f}ms | Loss: {float(loss):.4f} - Acc: {float(acc)*100:.1f}%")

avg_d_ms = np.mean(data_times) * 1000
avg_c_ms = np.mean(comp_times) * 1000
tot_ms_sample = (avg_d_ms + avg_c_ms) / 32

print("\n" + "=" * 80)
print(f"[*] Model Variable Device: {get_var_device(smoke_model.trainable_variables[0])}")
print(f"[*] TIMING SPLIT: Data Loading = {avg_d_ms/32:.2f} ms/sample ({avg_d_ms/(avg_d_ms+avg_c_ms)*100:.1f}%) | GPU Compute = {avg_c_ms/32:.2f} ms/sample ({avg_c_ms/(avg_d_ms+avg_c_ms)*100:.1f}%)")
print(f"[*] Total Training Throughput: {tot_ms_sample:.2f} ms/sample (Estimated Full Epoch: {(tot_ms_sample * 53401)/1000/60:.1f} minutes on GPU)")
print("=" * 80)

## 7. Full Training Pipeline (Phase 1 Head + Phase 2 Fine-Tuning)
Executes with compiled `@tf.function` GPU graph steps, dynamic class weighting, explicit optimizer slot variable initialization, and per-epoch Google Drive checkpoints.

In [ ]:
train_df = full_df[full_df["split"] == "train"].reset_index(drop=True)
val_df = full_df[full_df["split"] == "validation"].reset_index(drop=True)
test_df = full_df[full_df["split"] == "test"].reset_index(drop=True)

print(f"[*] Full Train Set:      {len(train_df):,} images")
print(f"[*] Full Validation Set: {len(val_df):,} images")
print(f"[*] Full Test Set:       {len(test_df):,} images")

BATCH_SIZE = 64
train_ds = create_tf_data_pipeline(train_df, BASE_DATA_DIR, batch_size=BATCH_SIZE, is_training=True)
val_ds = create_tf_data_pipeline(val_df, BASE_DATA_DIR, batch_size=BATCH_SIZE, is_training=False)
test_ds = create_tf_data_pipeline(test_df, BASE_DATA_DIR, batch_size=BATCH_SIZE, is_training=False)

# Maintain in-memory model from initialization/Phase 1
best_model_path = OUTPUT_DIR / "model.keras"
best_val_acc = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
opt_p1 = tf.keras.optimizers.Adam(1e-3)
opt_p2 = tf.keras.optimizers.Adam(1e-5)

@tf.function
def train_step_gpu(images, labels, opt):
    with tf.GradientTape() as tape:
        preds = model(images, training=True)
        loss_raw = loss_fn(labels, preds)
        weights = tf.gather(CLASS_WEIGHTS_T, labels)
        loss = tf.reduce_mean(loss_raw * weights)
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(preds, axis=1, output_type=tf.int32), labels), tf.float32))
    return loss, acc

@tf.function
def val_step_gpu(images, labels):
    preds = model(images, training=False)
    loss = loss_fn(labels, preds)
    acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(preds, axis=1, output_type=tf.int32), labels), tf.float32))
    return loss, acc

# ==========================================================================
# PHASE 1: Frozen Base Head Training (3 Epochs)
# ==========================================================================
PHASE1_EPOCHS = 3
print("=" * 80)
print(f"[*] STARTING PHASE 1: Training Head ({PHASE1_EPOCHS} Epochs, Base Frozen)")
print("=" * 80)
base_model.trainable = False

# Eagerly build opt_p1 slot variables before graph tracing
opt_p1.build(model.trainable_variables)
_ = train_step_gpu(tf.zeros((BATCH_SIZE, 224, 224, 3)), tf.zeros((BATCH_SIZE,), dtype=tf.int32), opt_p1)
print("[*] Phase 1 GPU Graph Trace ready.")

for epoch in range(PHASE1_EPOCHS):
    ep_start = time.time()
    tr_losses, tr_accs, data_times, comp_times = [], [], [], []
    ds_iter = iter(train_ds)
    batch_count = int(np.ceil(len(train_df) / BATCH_SIZE))
    
    for b in range(batch_count):
        t_d0 = time.time()
        X_b, y_b = next(ds_iter)
        t_data = time.time() - t_d0
        data_times.append(t_data)
        
        t_c0 = time.time()
        loss, acc = train_step_gpu(X_b, y_b, opt_p1)
        t_comp = time.time() - t_c0
        comp_times.append(t_comp)
        
        tr_losses.append(float(loss)); tr_accs.append(float(acc))
        if (b + 1) % 50 == 0 or (b + 1) == batch_count:
            print(f"  Phase 1 Epoch [{epoch+1}/{PHASE1_EPOCHS}] Batch [{b+1}/{batch_count}] - Data: {np.mean(data_times)*1000:4.1f}ms | GPU: {np.mean(comp_times)*1000:4.1f}ms | Loss: {float(loss):.4f} - Acc: {float(acc)*100:.1f}%", end="\r")

    val_losses, val_accs = [], []
    for X_b, y_b in val_ds:
        v_loss, v_acc = val_step_gpu(X_b, y_b)
        val_losses.append(float(v_loss)); val_accs.append(float(v_acc))

    mean_tr_loss, mean_tr_acc = np.mean(tr_losses), np.mean(tr_accs)
    mean_val_loss, mean_val_acc = np.mean(val_losses), np.mean(val_accs)
    ep_sec = time.time() - ep_start
    
    print(f"\n[Phase 1 Epoch {epoch+1}/{PHASE1_EPOCHS}] ({ep_sec:.1f}s) Train Loss: {mean_tr_loss:.4f}, Train Acc: {mean_tr_acc*100:.2f}% | Val Loss: {mean_val_loss:.4f}, Val Acc: {mean_val_acc*100:.2f}%")
    
    epoch_ckpt = OUTPUT_DIR / f"checkpoint_p1_epoch_{epoch+1}.keras"
    model.save(epoch_ckpt)
    if mean_val_acc > best_val_acc:
        best_val_acc = mean_val_acc
        model.save(best_model_path)
        print(f"  [+] Best model checkpoint saved to Google Drive: {best_model_path}")

# ==========================================================================
# PHASE 2: Fine-Tuning Top 30 Layers (3 Epochs, LR=1e-5)
# ==========================================================================
PHASE2_EPOCHS = 3
print("\n" + "=" * 80)
print(f"[*] STARTING PHASE 2: Fine-Tuning Top 30 Layers ({PHASE2_EPOCHS} Epochs, LR=1e-5)")
print("=" * 80)
base_model.trainable = True
for layer in base_model.layers[:-30]: layer.trainable = False
for layer in base_model.layers[-30:]: layer.trainable = True

# Explicitly build opt_p2's slot variables eagerly for the new unfrozen variables BEFORE tracing
opt_p2.build(model.trainable_variables)

# Trace graph for unfrozen weights
_ = train_step_gpu(tf.zeros((BATCH_SIZE, 224, 224, 3)), tf.zeros((BATCH_SIZE,), dtype=tf.int32), opt_p2)
print("[*] Phase 2 GPU Graph Trace ready with unfrozen layers.")

for epoch in range(PHASE2_EPOCHS):
    ep_start = time.time()
    tr_losses, tr_accs, data_times, comp_times = [], [], [], []
    ds_iter = iter(train_ds)
    batch_count = int(np.ceil(len(train_df) / BATCH_SIZE))
    
    for b in range(batch_count):
        t_d0 = time.time()
        X_b, y_b = next(ds_iter)
        t_data = time.time() - t_d0
        data_times.append(t_data)
        
        t_c0 = time.time()
        loss, acc = train_step_gpu(X_b, y_b, opt_p2)
        t_comp = time.time() - t_c0
        comp_times.append(t_comp)
        
        tr_losses.append(float(loss)); tr_accs.append(float(acc))
        if (b + 1) % 50 == 0 or (b + 1) == batch_count:
            print(f"  Phase 2 Epoch [{epoch+1}/{PHASE2_EPOCHS}] Batch [{b+1}/{batch_count}] - Data: {np.mean(data_times)*1000:4.1f}ms | GPU: {np.mean(comp_times)*1000:4.1f}ms | Loss: {float(loss):.4f} - Acc: {float(acc)*100:.1f}%", end="\r")

    val_losses, val_accs = [], []
    for X_b, y_b in val_ds:
        v_loss, v_acc = val_step_gpu(X_b, y_b)
        val_losses.append(float(v_loss)); val_accs.append(float(v_acc))

    mean_tr_loss, mean_tr_acc = np.mean(tr_losses), np.mean(tr_accs)
    mean_val_loss, mean_val_acc = np.mean(val_losses), np.mean(val_accs)
    ep_sec = time.time() - ep_start
    
    print(f"\n[Phase 2 Epoch {epoch+1}/{PHASE2_EPOCHS}] ({ep_sec:.1f}s) Train Loss: {mean_tr_loss:.4f}, Train Acc: {mean_tr_acc*100:.2f}% | Val Loss: {mean_val_loss:.4f}, Val Acc: {mean_val_acc*100:.2f}%")
    
    epoch_ckpt = OUTPUT_DIR / f"checkpoint_p2_epoch_{epoch+1}.keras"
    model.save(epoch_ckpt)
    if mean_val_acc > best_val_acc:
        best_val_acc = mean_val_acc
        model.save(best_model_path)
        print(f"  [+] Best model checkpoint saved to Google Drive: {best_model_path}")

## 8. Full Test Split Evaluation & Forensic Report
Evaluates the 9,000-image test split, plots confusion matrix, sub-dataset accuracy breakdown, and critical manipulation false-negative rate.

In [ ]:
# Load best checkpoint for test evaluation
if best_model_path.exists():
    model = tf.keras.models.load_model(best_model_path)
    print(f"[*] Loaded best model from {best_model_path} for final evaluation.")

y_true_all, y_pred_probs_all = [], []
for X_b, y_b in test_ds:
    probs = model(X_b, training=False)
    y_pred_probs_all.append(probs.numpy())
    y_true_all.append(y_b.numpy())

y_true = np.concatenate(y_true_all)
y_probs = np.concatenate(y_pred_probs_all)
y_preds = np.argmax(y_probs, axis=1)
overall_acc = np.mean(y_preds == y_true) * 100
target_names = [IDX_TO_CLASS[i] for i in range(3)]

print("=" * 80)
print(f"[*] OVERALL TEST SET ACCURACY: {overall_acc:.2f}%")
print("=" * 80)

# 1. Classification Report
print("\n--- Classification Report ---")
cls_report = classification_report(y_true, y_preds, target_names=target_names, digits=4, zero_division=0)
cls_report_dict = classification_report(y_true, y_preds, target_names=target_names, digits=4, output_dict=True, zero_division=0)
print(cls_report)

# 2. Confusion Matrix Plot
cm = confusion_matrix(y_true, y_preds, labels=[0, 1, 2])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title(f"EfficientNetB0 Authenticity Confusion Matrix (Test Acc: {overall_acc:.2f}%)")
plt.ylabel("True Class"); plt.xlabel("Predicted Class"); plt.tight_layout()
cm_file = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_file, dpi=150)
plt.show()
print(f"[*] Confusion matrix plot saved to Google Drive: {cm_file}")

# 3. ROC-AUC Curves
y_true_onehot = np.eye(3)[y_true]
roc_auc_val = float(roc_auc_score(y_true_onehot, y_probs, multi_class='ovr'))
print(f"[*] Multi-Class One-vs-Rest ROC-AUC: {roc_auc_val:.4f}")

plt.figure(figsize=(7, 6))
for i, c_name in enumerate(target_names):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_probs[:, i])
    plt.plot(fpr, tpr, label=f"{c_name} (AUC = {roc_auc_score(y_true_onehot[:, i], y_probs[:, i]):.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('Multi-Class ROC Curves')
plt.legend(); plt.tight_layout()
roc_file = OUTPUT_DIR / "roc_curves.png"
plt.savefig(roc_file, dpi=150)
plt.show()

# 4. Sub-Dataset Breakdown
print("\n--- Test Accuracy Breakdown by Source Dataset ---")
test_df_copy = test_df.copy()
test_df_copy["y_true"] = y_true
test_df_copy["y_pred"] = y_preds
test_df_copy["correct"] = test_df_copy["y_true"] == test_df_copy["y_pred"]

sub_metrics = {}
for src_ds, grp in test_df_copy.groupby("source_dataset"):
    ds_acc = float(grp["correct"].mean() * 100)
    sub_metrics[src_ds] = {"accuracy_pct": round(ds_acc, 2), "correct": int(grp["correct"].sum()), "total": len(grp)}
    print(f"   * {src_ds:<35}: {ds_acc:6.2f}% ({grp['correct'].sum()}/{len(grp)} samples)")

# 5. High-Stakes Analysis: AI_MODIFIED <-> REAL
manip_called_real = int(cm[1][2])
real_called_manip = int(cm[2][1])
total_manip = int(np.sum(cm[1]))
fn_rate = round((manip_called_real / max(1, total_manip)) * 100, 2)

print("\n" + "!" * 80)
print("      HIGH-STAKES CRITICAL ANALYSIS: AI_MODIFIED <-> REAL CONFUSION")
print("!" * 80)
print(f" [!] False Negatives (AI_MODIFIED predicted as REAL) : {manip_called_real} / {total_manip} ({fn_rate}%)")
print(f" [!] False Positives (REAL predicted as AI_MODIFIED) : {real_called_manip} / {int(np.sum(cm[2]))}")
print(f" [!] Critical Risk: Missing a modified facial deepfake is the highest-consequence error.")
print("!" * 80)

## 9. Export Final Metadata & Reports to Google Drive

In [ ]:
metadata = {
    "model_name": "EfficientNetB0_Authenticity_3Class",
    "classes": ["ai_generated", "ai_modified", "real"],
    "class_to_idx": CLASS_TO_IDX,
    "input_size": 224,
    "preprocessing": "efficientnet",
    "face_crop_sources": ["FaceForensics++"],
    "class_weights": CLASS_WEIGHTS,
    "metrics": {
        "overall_accuracy_pct": round(overall_acc, 2),
        "roc_auc_ovr": round(roc_auc_val, 4),
        "high_stakes_false_negative_rate_pct": fn_rate,
        "subdataset_accuracy": sub_metrics,
        "classification_report": cls_report_dict
    }
}

meta_file = OUTPUT_DIR / "metadata.json"
with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

eval_file = OUTPUT_DIR / "evaluation_report.json"
with open(eval_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ All training outputs successfully exported to Google Drive: {OUTPUT_DIR}")
!ls -la "/content/drive/MyDrive/deepfake_project/outputs"